## 定义所需的库

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from IPython.display import display, clear_output
import ipywidgets as widgets
import time
from collections import deque

## 加载图像
在本节中，将外部图像文件加载到程序中以表示游戏的视觉元素。
使用了三张图像：一张代表船，一张代表传教士（人类），一张代表食人族（僵尸）。
使用 matplotlib.image.imread 读取这些图像，这会将图像文件转换为可以在图表中显示的数组。
与使用简单的文本或形状相比，这种方法使游戏具有更直观和吸引人的界面。

In [ ]:
boat_img = mpimg.imread("boat.png")
human_img = mpimg.imread("human.png")
zombie_img = mpimg.imread("zombie.png")

## 游戏状态
游戏状态定义了游戏在任何时间点的当前配置。
它跟踪河流左右两侧的传教士和食人族的数量，并使用变量 boat_x 记录船的位置。
在每次有效移动后，此状态都会更新，并用于确定游戏的进度。
保持清晰的游戏状态可确保始终如一地执行游戏规则。

In [ ]:
boat_x = 6.5  # single source of truth

missionary_on_right = 3
cannibals_on_right = 3

missionary_on_left = 0
cannibals_on_left = 0

step = 0
output = widgets.Output()

## 图像辅助函数
图像辅助函数用于在图表的特定位置放置和显示角色图像（传教士和食人族）。
它使用 OffsetImage 和 AnnotationBbox 将图像以定义的大小插入到给定的坐标处。
这使得图像能够精确地定位在游戏场景中。
通过使用这个辅助函数，代码避免了重复，并保持了可视化过程的整洁和有序。

In [ ]:
def add_image(ax, img, x, y, zoom=0.08):
    imagebox = OffsetImage(img, zoom=zoom)
    ab = AnnotationBbox(imagebox, (x, y), frameon=False)
    ax.add_artist(ab)

## 绘制场景
draw_scene 函数负责在每一步渲染整个游戏环境。
它根据当前的游戏状态绘制河流、两块陆地、船以及所有的传教士和食人族。
该函数使用存储的变量值将每个元素正确地放置在屏幕上。
这确保了视觉表现始终与游戏状态保持同步。

In [ ]:
def draw_scene():
    fig, ax = plt.subplots(figsize=(8,2))
    
    # River
    ax.add_patch(plt.Rectangle((0,0), 10, 2, color='lightblue'))
    
    # Land
    ax.add_patch(plt.Rectangle((0,0), 3, 2, color='lightgreen'))
    ax.add_patch(plt.Rectangle((7,0), 3, 2, color='lightgreen'))
    
    # Boat (always use boat_x)
    imagebox = OffsetImage(boat_img, zoom=0.15)
    ab = AnnotationBbox(imagebox, (boat_x, 1.0), frameon=False)
    ax.add_artist(ab)
    
    # LEFT SIDE
    for i in range(missionary_on_left):
        add_image(ax, human_img, 0.7, 1.6 - i*0.3)
    for i in range(cannibals_on_left):
        add_image(ax, zombie_img, 1.7, 1.6 - i*0.3)
    
    # RIGHT SIDE
    for i in range(missionary_on_right):
        add_image(ax, human_img, 8.3, 1.6 - i*0.3)
    for i in range(cannibals_on_right):
        add_image(ax, zombie_img, 7.3, 1.6 - i*0.3)
    
    ax.set_xlim(0,10)
    ax.set_ylim(0,2)
    ax.axis('off')
    if boat_x ==6.5:
        boat_pos = '左侧'
    else:
        boat_pos = '右侧'
    plt.title(f"船的位置: {boat_pos} | 步数 {step}")
    plt.show()

## 动画
动画函数控制船只在河流左右两侧之间的平滑移动。
它在几个步骤中逐渐更新船的位置，从而产生运动效果，而不是瞬间跳跃。
在动画过程中，场景会不断重绘以反映更新后的船只位置。
这为状态之间提供了更真实、更具视觉吸引力的过渡。

In [ ]:
def animate_boat(start_x, end_x):
    global boat_x
    
    steps = 12
    
    for i in range(steps + 1):
        boat_x = start_x + (end_x - start_x) * i / steps
        
        # Clamp strictly to water
        boat_x = max(3.5, min(6.5, boat_x))
        
        with output:
            clear_output(wait=True)
            draw_scene()
        
        time.sleep(0.04)

## 游戏逻辑
游戏逻辑控制玩家如何与游戏交互，并确保所有移动都遵循定义的规则。
它检查所选的传教士和食人族数量是否有效，并根据船的移动更新他们的位置。
该逻辑还通过评估每侧传教士和食人族的数量来确定一步移动是否会导致胜利或失败。
这确保了游戏正确进行并强制执行问题的约束条件。

In [ ]:
def play_move(missionary, cannibals):
    global missionary_on_right, cannibals_on_right
    global missionary_on_left, cannibals_on_left
    global boat_x, step
    
    with output:
        clear_output(wait=True)
        
        if (missionary + cannibals) not in [1,2]:
            print("❌ 无效移动")
            draw_scene()
            return
        
        # RIGHT → LEFT
        if boat_x > 5:
            if missionary > missionary_on_right or cannibals > cannibals_on_right:
                print("❌ 无效移动")
                draw_scene()
                return
            
            animate_boat(6.5, 3.5)
            
            missionary_on_right -= missionary
            cannibals_on_right -= cannibals
            missionary_on_left += missionary
            cannibals_on_left += cannibals
        
        # LEFT → RIGHT
        else:
            if missionary > missionary_on_left or cannibals > cannibals_on_left:
                print("❌ 无效移动")
                draw_scene()
                return
            
            animate_boat(3.5, 6.5)
            
            missionary_on_right += missionary
            cannibals_on_right += cannibals
            missionary_on_left -= missionary
            cannibals_on_left -= cannibals
        
        step += 1
        
        draw_scene()
        
        # LOSE
        if ((missionary_on_right != 0 and missionary_on_right < cannibals_on_right) or 
            (missionary_on_left != 0 and missionary_on_left < cannibals_on_left)):
            print("💀 你输了")
            return
        
        # WIN
        if missionary_on_left == 3 and cannibals_on_left == 3:
            print("🎉 你赢了")
            return


## 创建求解器按钮
使用 BFS（广度优先搜索）算法提供解决方案

In [ ]:
def is_valid_state(M_left, C_left, M_right, C_right):
    if M_left < 0 or C_left < 0 or M_right < 0 or C_right < 0:
        return False
    
    if (M_left != 0 and M_left < C_left):
        return False
    if (M_right != 0 and M_right < C_right):
        return False
    
    return True


def bfs_solve():
    global missionary_on_left, cannibals_on_left
    global missionary_on_right, cannibals_on_right
    global boat_side, boat_x
    
    start = (3, 3, "Right")
    goal = (0, 0, "Left")
    
    queue = deque()
    visited = set()
    
    # (state, path)
    queue.append((start, []))
    
    moves = [(1,0), (2,0), (0,1), (0,2), (1,1)]
    
    while queue:
        (ML, CL, side), path = queue.popleft()
        
        if (ML, CL, side) in visited:
            continue
        
        visited.add((ML, CL, side))
        
        if (ML, CL, side) == goal:
            return path
        
        for m, c in moves:
            
            if side == "Right":
                new_state = (ML - m, CL - c, "Left")
                new_ML = ML - m
                new_CL = CL - c
                new_MR = 3 - new_ML
                new_CR = 3 - new_CL
                
                # invalid move check
                if m > ML or c > CL:
                    continue
                
            else:
                new_state = (ML + m, CL + c, "Right")
                new_ML = ML + m
                new_CL = CL + c
                new_MR = 3 - new_ML
                new_CR = 3 - new_CL
                
                if m > (3 - ML) or c > (3 - CL):
                    continue
            
            if not is_valid_state(new_ML, new_CL, new_MR, new_CR):
                continue
            
            if new_state not in visited:
                queue.append((new_state, path + [(m, c, side)]))
    
    return None


def solve_game(b):
    solution = bfs_solve()
    
    if solution is None:
        print("未找到解决方案")
        return
    
    print("求解中...")
    
    for m, c, side in solution:
        play_move(m, c)
        time.sleep(1)

## 用户界面设计
UI 设计通过使用按钮选择不同的移动组合，为玩家提供了一个交互式界面。
每个按钮对应一个有效的动作，例如将一名传教士或一名食人族运过河。
还包括一个重置按钮，以便随时重新开始游戏。
与手动输入相比，这种设计使游戏更容易玩，也更人性化。

In [ ]:
moves = {
    "1M": (1,0),
    "2M": (2,0),
    "1C": (0,1),
    "2C": (0,2),
    "1M1C": (1,1)
}

buttons = []

for label, (m,c) in moves.items():
    btn = widgets.Button(description=label)
    
    def on_click(b, m=m, c=c):
        play_move(m, c)
    
    btn.on_click(on_click)
    buttons.append(btn)

def reset_game(b=None):
    global boat_x, missionary_on_right, cannibals_on_right
    global missionary_on_left, cannibals_on_left, step
    
    boat_x = 6.5
    
    missionary_on_right = 3
    cannibals_on_right = 3
    missionary_on_left = 0
    cannibals_on_left = 0
    step = 0
    
    with output:
        clear_output()
        draw_scene()

reset_btn = widgets.Button(description="🔄 重置")
reset_btn.on_click(reset_game)
auto_btn = widgets.Button(description="🤖 自动求解")
auto_btn.on_click(solve_game)

## 显示：运行模拟
显示部分负责在 Jupyter Notebook 中渲染用户界面元素和游戏输出。
它显示了交互式按钮和输出区域，在每次移动后，游戏场景都会在此区域更新。
IPython.display 的使用确保了视觉组件的动态刷新。
这允许玩家在笔记本环境中实时与游戏进行交互。

In [ ]:
display(widgets.HBox(buttons))
display(reset_btn)
#display(auto_btn)
display(output)
with output:
    draw_scene()

## 任务 3 结束
请写下你今天从本教程中学到了什么